## 3DoF Entry VTOL w/o Aoa SCP

Imports

In [ ]:
# Basic imports
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Trajopt imports --> pip install -e ~/ACL/trajopt
import trajopt; importlib.reload(trajopt)
import trajopt.core.modules.method.scp as scp
import trajopt.core.problem as prob
import trajopt.utils.config_loader as cfg
import trajopt.utils.tools as tools
import trajopt.analysis.default_analysis as default_analysis
import trajopt.analysis.statistics as stats
import trajopt.core.modules.analysis.monte_carlo as mc
import copy

from trajopt.analysis.trajplots import *

from custom_functions_dan import max_q_nonjax, max_Q_nonjax, max_load_nonjax, terminal_cost



setup problem and run SCP

In [ ]:
example_name = "vtol1_entry_3dof"
nominal_config  = cfg.load_configs(example_name)

# either generate mc variations from yamls or load existing saved mc variations
gen_mc_variations    = 1

# save mc variations to file (specificy a name for this set of mc variations as well)
save_mc_variations   = 0
mc_name = "mc1"

# save scenario data to file (save the scenario data struct)
save_scenario_data   = 0

# run mc analysis loop
# TODO(Carlos and  Dan):  shouldn't need to return problem, just scenario data, something below depends on it though (this is the most recent problem instance)
scenario_data, problem = mc.run_mc_analysis(example_name, nominal_config, gen_mc_variations, save_mc_variations, save_scenario_data, mc_name)

mc analysis

In [ ]:
file = '~/masters-link/ACL/prototypes/trajopt/src/trajopt/examples/vtol1_entry_3dof/data/statistics/mc1.txt' # Example file
analysis = stats.analyze_quality_metrics(scenario_data, filename=file) # Can run without filename if you don't want to save LaTeX tables

make plots

In [ ]:
# print(scenario_data['autotune'])
data = {'scenario1':scenario_data}
temp = data['scenario1']['autotune']['mc_data'][0]['iters'][0]
print(list(temp))

In [ ]:
PLTS1 = SCVXPLOTS(data);
PLTS1.setCurrent({'scenarios':['scenario1'],
                  'methods':['standard','autotune'],
                  'runs':list(range(1000)),
                  'iters':list(range(1000))[1:]})

# state_inds = [0,1,2];
tags = ['max_q','max_Q','max_load','terminal_cost'];
for tag in tags:
    tag1 = tag + '_sub'; tag2 = tag + '_nl';
    func_args1 = ['ts','zs',None,problem];
    func_args2 = ['t_nl','z_nl',None,problem];
    if tag == 'max_q': func = max_q_nonjax
    if tag == 'max_Q': func = max_Q_nonjax
    if tag == 'max_load': func = max_load_nonjax
    if tag == 'terminal_cost': func = terminal_cost;

    PLTS1.calcField(tag1,func,func_args = func_args1)
    PLTS1.calcField(tag2,func,func_args = func_args2)
    



# if True: 
    
#     tag2 = 'max_Q';    func_args = ['ts','zs',None,problem];
#     tag3 = 'max_load'; func_args = ['ts','zs',None,problem];
#     func1 = max_q_nonjax;
#     func2 = max_Q_nonjax;
#     func3 = max_load_nonjax;

#     PLTS1.calcField(tag1,func1,func_args = func_args)
#     PLTS1.calcField(tag2,func2,func_args = func_args)
#     PLTS1.calcField(tag3,func3,func_args = func_args)
else:
    def testFunc1(x,y): return np.random.rand(1)[0];
    def testFunc2(x,y): return np.random.rand(1)[0];
    def testFunc3(x,y): return np.random.rand(1)[0];
    func_args = ['z_nl','z_nl'];
    tag1 = 'max_q';
    tag2 = 'max_Q'; 
    tag3 = 'max_load'; 

    PLTS1.calcField(tag1,testFunc1,func_args = func_args)
    PLTS1.calcField(tag2,testFunc2,func_args = func_args)
    PLTS1.calcField(tag3,testFunc2,func_args = func_args)



In [ ]:

PENS = {};
# PENS['z'] = {'frgba':[0,0,0,0.1],'lrgba':[0,0,0,0.1],'lw':2,'ls':'-','msty':'','msz':4};
PENS['init'] = {'frgba':[.0,.0,.0,.1],'lrgba':[.0,.0,.0,1.],'lw':1,'ls':'--','msty':'' ,'msz':3};
PENS['itr']  = {'frgba':[.0,.0,.0,.1],'lrgba':[.0,.0,1.,.2],'lw':1,'ls':'--','msty':'' ,'msz':3};
PENS['opt']  = {'frgba':[.0,.0,.0,.1],'lrgba':[.0,.0,1.,1.],'lw':1,'ls':''  ,'msty':'o','msz':3};
PENS['prop'] = {'frgba':[.0,.0,.0,.1],'lrgba':[1.,.0,.0,1.],'lw':1,'ls':'-' ,'msty':'' ,'msz':3};
PENS['ref']  = {'frgba':[.0,.0,.0,.1],'lrgba':[.0,.0,.0,1.],'lw':1,'ls':'--','msty':'*','msz':3};

PENS['standard']  = {'frgba':[.0,.0,.0,.1],'lrgba':[.0,.0,.1,1.],'lw':1,'ls':'-','msty':'o','msz':3};
PENS['autotune']  = {'frgba':[.0,.0,.0,.1],'lrgba':[.5,.0,.1,1.],'lw':1,'ls':'-','msty':'o','msz':3};


# Fig.5 Bank Angle vs Time (for AutoScvx)

In [ ]:
### TO FIX
### ================================================ ###
sind = 0; # STATE INDEX FOR BANK ANGLE 
### ================================================ ###

# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.5,0.5,0.9,0.9];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [sind];
state_plot_inds = {sind:(0,0)};
state_names = {sind:'Bank Angle'};

titles = {}; ylabels = {};
titles[(0,0)] = 'Bank Angle vs. Time';
ylabels[(0,0)] = 'Bank Angle, $\sigma$ [deg]';

scenarios = ['scenario1'];
methods = ['autotune']; # methods = ['standard','autotune'];
lgnd = 'Fig5';

itrs_all = list(range(1000))[2:]; runs = list(range(1000));
for sind in state_inds:
    aind = state_plot_inds[sind];
    ax = axs[aind];
    PLTS1.setCurrent({'scenarios':scenarios,'methods':methods,'runs':runs})

    params1 = {'label':'Initial guess','x':'t','y':('u',sind),'iters':[1],'legend':lgnd};
    params2 = {'label':'iterations','x':'t','y':('u',sind),'iters':itrs_all,'legend':lgnd};
    params3 = {'label':'Propogated','x':'t_nl','y':('u_nl',sind),'iters':[-1],'legend':lgnd};
    params4 = {'label':'Optimal Solution','x':'t','y':('u',sind),'iters':[-1],'legend':lgnd};

    PLTS1.addPlot2D(ax,pen=PENS['init'],ins=params1);
    PLTS1.addPlot2D(ax,pen=PENS['itr'] ,ins=params2);
    PLTS1.addPlot2D(ax,pen=PENS['prop'],ins=params3); 
    PLTS1.addPlot2D(ax,pen=PENS['opt'] ,ins=params4); 
    
    params = {};
    params['title'] = {'text':titles[aind],'fontsize':20}
    params['xlabel'] = {'label':'Time [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[aind],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

# Fig.6

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid2D = {}; grid3D = {};
grid3D[(0,0)] = [0.05,0.05,0.6,0.9]; 
grid2D[(0,1)] = [0.70,0.05,0.2,0.9];

axs1 = PLTS1.createGrid(fig,grid = grid2D);
# axs2 = PLTS1.createGrid(fig,grid = grid3D,ins={'plt_typ':'3d'});
axs2 = {(0,0): fig.add_axes(grid3D[(0,0)],projection='3d')} # colorbar axis]
axs = {**axs1,**axs2};

state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};

titles = {}; ylabels = {}; xlabels = {}; zlabels = {};
titles[0] = 'Position(3D) vs Time';
titles[1] = 'Position(2D) vs Time';

xlabels[0] = 'Latitude $\phi$ [deg]';
ylabels[0] = 'Longitude $\\theta$ [deg]';
zlabels[0] = 'Altitude $h$ [km]';

ylabels[1] = 'Latitude $\phi$ [deg]';
xlabels[1] = 'Longitude $\\theta$ [deg]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig6'
j = 1;
itrs_all = list(range(1000))[2:]; 
runs = [0]; #list(range(1000)); itrs = [-1];
ax = axs[state_plot_inds[j]];

ax = axs[(0,1)]; #state_plot_inds[j]];
sindx = 1; sindy = 2; j = 1;
# itrs_all = list(range(1000))[2:]; 
runs = [0]; itrs = [0]; 
PLTS1.setCurrent({'scenarios':scenarios,'methods':methods,'runs':runs})
params1 = {'label':'Initial guess','x':('z',sindx),'y':('z',sindy),'iters':[1],'legend':lgnd,};
params2 = {'label':'iterations','x':('z',sindx),'y':('z',sindy),'iters':itrs_all,'legend':lgnd};
params3 = {'label':'Propogated','x':('z_nl',sindx),'y':('z_nl',sindy),'iters':[-1],'legend':lgnd};
params4 = {'label':'Optimal Solution','x':('z',sindx),'y':('z',sindy),'iters':[-1],'legend':lgnd};

PLTS1.addPlot2D(ax,pen=PENS['init'],ins=params1);
PLTS1.addPlot2D(ax,pen=PENS['itr'] ,ins=params2);
PLTS1.addPlot2D(ax,pen=PENS['prop'],ins=params3); 
PLTS1.addPlot2D(ax,pen=PENS['opt'] ,ins=params4);     

params = {};
params['title'] = {'text':titles[1],'fontsize':20}
params['xlabel'] = {'label':xlabels[1],'fontsize':16}
params['ylabel'] = {'label':ylabels[1],'fontsize':16}
params['ticks'] = {'labelsize':20,'width':2};
PLTS1.setParams(ax,params);
PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});


ax = axs[(0,0)];
sindx = 1; sindy = 2; sindz = 0; 

params1 = {'label':'Initial guess','x':('z',sindx),'y':('z',sindy),'z':('z',sindz),'iters':[1],'legend':lgnd,};
params2 = {'label':'iterations','x':('z',sindx),'y':('z',sindy),'z':('z',sindz),'iters':itrs_all,'legend':lgnd};
params3 = {'label':'Propogated','x':('z_nl',sindx),'y':('z_nl',sindy),'z':('z_nl',sindz),'iters':[-1],'legend':lgnd};
params4 = {'label':'Optimal Solution','x':('z',sindx),'y':('z',sindy),'z':('z',sindz),'iters':[-1],'legend':lgnd};

PLTS1.addPlot3D(ax,pen=PENS['init'],ins=params1);
PLTS1.addPlot3D(ax,pen=PENS['itr'] ,ins=params2);
PLTS1.addPlot3D(ax,pen=PENS['prop'],ins=params3); 
PLTS1.addPlot3D(ax,pen=PENS['opt'] ,ins=params4);     

params = {};
params['title'] = {'text':titles[0],'fontsize':20}
params['xlabel'] = {'label':xlabels[0],'fontsize':16}
params['ylabel'] = {'label':ylabels[0],'fontsize':16}
params['zlabel'] = {'label':zlabels[0],'fontsize':16}
params['ticks'] = {'labelsize':20,'width':2};
PLTS1.setParams(ax,params);
PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});






# Fig. 7

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(1,1)] = [0.55,0.05,0.4,0.35];
grid[(0,1)] = [0.55,0.6,0.4,0.35];
grid[(1,0)] = [0.05,0.05,0.4,0.35];
grid[(0,0)] = [0.05,0.6,0.4,0.35];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [0,3,4,5] # replace with appropriate state indices
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};



state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};


titles = {}; ylabels = {};
titles[2] = 'Flight Path Angle vs Time';
titles[3] = 'Heading vs Time';
titles[0] = 'Attitude vs Time';
titles[1] = 'Velocity vs Time';

ylabels[2] = 'Flight Path Angle [deg]';
ylabels[3] = 'Heading $\psi$ [deg]';
ylabels[0] = 'Attitude [km]';
ylabels[1] = 'Velocity [$m/s$]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig7';

itrs_all = list(range(1000))[2:]; runs = list(range(1000));
for j,sind in enumerate(state_inds):
    print(sind)
    ax = axs[state_plot_inds[j]];
    PLTS1.setCurrent({'scenarios':scenarios,'methods':methods,'runs':runs})

    params1 = {'label':'Initial guess','x':'t','y':('z',sind),'iters':[1],'legend':lgnd,};
    params2 = {'label':'iterations','x':'t','y':('z',sind),'iters':itrs_all,'legend':lgnd};
    params3 = {'label':'Propogated','x':'t_nl','y':('z_nl',sind),'iters':[-1],'legend':lgnd};
    params4 = {'label':'Optimal Solution','x':'t','y':('z',sind),'iters':[-1],'legend':lgnd};

    PLTS1.addPlot2D(ax,pen=PENS['init'],ins=params1);
    PLTS1.addPlot2D(ax,pen=PENS['itr'] ,ins=params2);
    PLTS1.addPlot2D(ax,pen=PENS['prop'],ins=params3); 
    PLTS1.addPlot2D(ax,pen=PENS['opt'] ,ins=params4); 
    
    params = {};
    params['title'] = {'text':titles[j],'fontsize':20}
    params['xlabel'] = {'label':'Time [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[j],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    if j == 3: PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

In [ ]:
# temp = PLTS1.data['scenario1']['autotune']['mc_data'][0]['iters'][2]['max_load']
# print(temp)
# # print(list(temp))
# # print(temp['max_q'])

# Fig.8 heat rate etc. 

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.05,0.05,0.3,0.9];
grid[(0,1)] = [0.4,0.05,0.3,0.9];
grid[(0,2)] = [0.75,0.05,0.3,0.9];

axs = PLTS1.createGrid(fig,grid = grid);

tags = ['max_Q','max_q','max_load']
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};



state_plot_inds = {0:(0,0),1:(0,1),2:(0,2)};


titles = {}; ylabels = {};

titles[0] = 'Heat Rate Constraint';
titles[1] = 'Dynamic Pressure Constraint';
titles[2] = 'Normal Load Constraint';

ylabels[0] = 'Heating Rate [kW/$m^2$]';
ylabels[1] = 'Dynamic Pressure [kPa]';
ylabels[2] = 'Normal Load [$g\'s$]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig8';
itrs_all = list(range(1000))[1:]; runs = runs = list(range(1000));
for j,tag in enumerate(tags):
    
    ax = axs[state_plot_inds[j]];
    PLTS1.setCurrent({'scenarios':scenarios,'methods':methods,'runs':runs})

    # params1 = {'label':'Initial guess','x':'t','y':tag,'iters':[1],'legend':lgnd,};
    params2 = {'label':'iterations','x':'ts','y':tag + '_sub','iters':itrs_all,'legend':lgnd};
    params3 = {'label':'Propogated','x':'t_nl','y':tag + '_nl','iters':[-1],'legend':lgnd};
    params4 = {'label':'Optimal Solution','x':'ts','y':tag + '_sub','iters':[-1],'legend':lgnd};

    # PLTS1.addPlot2D(ax,pen=PENS['init'],ins=params1);
    PLTS1.addPlot2D(ax,pen=PENS['itr'] ,ins=params2);
    PLTS1.addPlot2D(ax,pen=PENS['prop'],ins=params3); 
    PLTS1.addPlot2D(ax,pen=PENS['opt'] ,ins=params4); 
    
    params = {};
    params['title'] = {'text':titles[j],'fontsize':20}
    params['xlabel'] = {'label':'Time [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[j],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

In [ ]:
# temp = data['scenario1']['autotune']['mc_data'][0]['iters'][2]['weights'];#params']['method']['weights']['W_dyn'];
temp = data['scenario1']['autotune']['mc_data'][0]['iters'][2]['conv_data'];#['chk_feas_ineq'];#params']['method']['weights']['W_dyn'];
# ['w_cost', 'alpha_z', 'alpha_u', 'beta', 'gamma', 'eps_nonzero1',
#  'eps_nonzero2', 'wbuff', 'w_path_scale', 'w_custom_scale', 'w_nfz_scale',
#  'w_dyn_scale', 'w_term_scale',

#  'W_ineq', 'W_term', 'W_dyn', 'W_plus','W_minus',
#  'dual_ineq', 'dual_term', 'dual_dyn', 'dual_plus', 'dual_minus',

#  'wtr_z', 'wtr_u', 'w_fac_N', 'w_fac_Nm1', 'w_ctcs', 'data']


# print(temp)
# print(list(temp))
# print(temp['W_ineq'])

In [ ]:

temp = data['scenario1']['autotune']['mc_data'][0]['iters'][2];#['weights'];
print(temp['weights']['W_ineq'])



# Fig.9 

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(1,1)] = [0.55,0.05,0.4,0.35];
grid[(0,1)] = [0.55,0.6,0.4,0.35];
grid[(1,0)] = [0.05,0.05,0.4,0.35];
grid[(0,0)] = [0.05,0.6,0.4,0.35];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [0,3,4,5] # replace with appropriate state indices
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};

state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};

titles = {}; ylabels = {};
titles[0] = ''; titles[1] = ''; titles[2] = ''; titles[3] = '';

ylabels[0] = 'No-fly zone quadratic penalty weights';
ylabels[1] = 'Path constraint quadratic penalty weights';
ylabels[2] = 'No-fly zone linear penalty weights';
ylabels[3] = 'Path constraint linear penalty weights';



weights = ['W_ineq','dual_ineq']; #'W_term','W_dyn']; #,'W_plus','W_minus']
# 'W_ineq' -> path constraints
# 'W_term' -> terminal condition
# 'W_dyn', -> dynamics
# 'W_plus', 'W_minus', -> the weird quadratic 1-norm 
# 'dual_ineq', 'dual_term', 'dual_dyn', 'dual_plus', 'dual_minus', <- dual versions

scenarios = ['scenario1'];
methods = ['standard','autotune'];
lgnd = 'Fig9';

itrs_all = list(range(1000))[1:]; runs = list(range(1000));
for j,weight in enumerate(weights):
    print(weight)
    ax = axs[state_plot_inds[j]];
    PLTS1.setCurrent({'scenarios':scenarios,'methods':methods,'runs':runs})

    params1 = {'label':'Initial guess','x':'t','y':weight,'iters':[1],'legend':lgnd,'dataloc':'weights'};
    params2 = {'label':'iterations','x':'t','y':weight,'iters':itrs_all,'legend':lgnd,'dataloc':'weights'};
    # params3 = {'label':'Propogated','x':'t_nl','y':weight,'iters':[-1],'legend':lgnd,'dataloc':'weights'};
    params4 = {'label':'Optimal Solution','x':'t','y':weight,'iters':[-1],'legend':lgnd,'dataloc':'weights'};

    # try: 
    PLTS1.addPlot2D(ax,pen=PENS['init'],ins=params1);
    PLTS1.addPlot2D(ax,pen=PENS['itr'] ,ins=params2);
    # PLTS1.addPlot2D(ax,pen=PENS['prop'],ins=params3); 
    PLTS1.addPlot2D(ax,pen=PENS['opt'] ,ins=params4); 
    # except: pass
    
    params = {};
    params['title'] = {'text':titles[j],'fontsize':20}
    params['xlabel'] = {'label':'Time [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[j],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    if j == 3: PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

# Fig.11

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.05,0.05,0.9,0.9];

axs = PLTS1.createGrid(fig,grid = grid);
sinds = [3] ;#tags = ['max_q','max_Q','max_load']
state_names = {0:'Term. Altitude',
               1:'Term. Longitude',
               2:'Term. Latitude',
               3:'Term. Flight Path Angle',
               4:'Term. Heading'}


lgnd = 'Fig11';
state_plot_inds = {0:(0,0),1:(0,1),2:(0,2)};

titles = {}; ylabels = {};
titles[0] = 'Penalty weights for constraints';
ylabels[0] = 'Terminal state constraint, quadratic penalty weights';

scenarios = ['scenario1'];
methods = ['autotune'];

itrs_all = list(range(1000))[1:]; runs = [0];#list(range(1000));
for j,sind in enumerate(sinds):
    ax = axs[state_plot_inds[0]];
    PLTS1.setCurrent({'scenarios':scenarios,'methods':methods,'runs':runs})

    params1 = {'label':'Initial guess','tinds':[-1],'y':('z',sind),'iters':[1],'legend':lgnd};
    params2 = {'label':'Iterations','tinds':[-1],'y':('z',sind),'iters':itrs_all,'legend':lgnd};
    params3 = {'label':'Propogated','tinds':[-1],'y':('z_nl',sind),'iters':[-1],'legend':lgnd};
    params4 = {'label':'Optimal Solution','tinds':[-1],'y':('z',sind),'iters':itrs_all,'legend':lgnd};
    PLTS1.addPlot2DIter(ax,pen=PENS['init'] ,ins=params1); 
    PLTS1.addPlot2DIter(ax,pen=PENS['itr'] ,ins=params2); 
    PLTS1.addPlot2DIter(ax,pen=PENS['prop'] ,ins=params3); 
    PLTS1.addPlot2DIter(ax,pen=PENS['opt'] ,ins=params4); 
    
    params = {};
    params['title'] = {'text':titles[0],'fontsize':20}
    params['xlabel'] = {'label':'Iterations [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[0],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

# Fig.12

In [ ]:
### TO FIX
### ================================================ ###
sind = 0; # STATE INDEX FOR BANK ANGLE 
### ================================================ ###

# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.5,0.5,0.9,0.9];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [sind];
state_plot_inds = {sind:(0,0)};
state_names = {sind:'Bank Angle'};

titles = {}; ylabels = {};
titles[(0,0)] = 'Bank Angle vs. Time';
ylabels[(0,0)] = 'Bank Angle, $\sigma$ [deg]';

scenarios = ['scenario1'];
methods = ['standard','autotune']; # methods = ['standard','autotune'];

lgnd = 'Fig12';
itrs_all = list(range(1000))[2:]; runs = [0];#list(range(1000));
for method in methods:
    aind = state_plot_inds[sind];
    ax = axs[aind];
    PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})

    params4 = {'label':method,'x':'t','y':('u',sind),'iters':[-1],'legend':lgnd};
    PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params4); 
    
    params = {};
    params['title'] = {'text':titles[aind],'fontsize':20}
    params['xlabel'] = {'label':'Time [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[aind],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

# Fig.13

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid2D = {}; grid3D = {};
grid3D[(0,0)] = [0.05,0.05,0.6,0.9]; 
grid2D[(0,1)] = [0.70,0.05,0.2,0.9];


axs1 = PLTS1.createGrid(fig,grid = grid2D);
# axs2 = PLTS1.createGrid(fig,grid = grid3D,ins={'plt_typ':'3d'});
axs2 = {(0,0): fig.add_axes(grid3D[(0,0)],projection='3d')} # colorbar axis]
axs = {**axs1,**axs2};

state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};

titles = {}; ylabels = {}; xlabels = {}; zlabels = {};
titles[0] = 'Position(3D) vs Time';
titles[1] = 'Position(2D) vs Time';

xlabels[0] = 'Latitude $\phi$ [deg]';
ylabels[0] = 'Longitude $\\theta$ [deg]';
zlabels[0] = 'Altitude $h$ [km]';

ylabels[1] = 'Latitude $\phi$ [deg]';
xlabels[1] = 'Longitude $\\theta$ [deg]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig13'
j = 1;
itrs_all = list(range(1000))[2:]; 
runs = [0]; #list(range(1000)); itrs = [-1];
ax = axs[state_plot_inds[j]];
for method in methods:
    ax = axs[(0,1)]; #state_plot_inds[j]];
    sindx = 1; sindy = 2;
    PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})
    params1 = {'label':method,'x':('z',sindx),'y':('z',sindy),'iters':[-1],'legend':lgnd};
    PLTS1.addPlot2D(ax,pen=PENS[method],ins=params1);

    params = {};
    params['title'] = {'text':titles[1],'fontsize':20}
    params['xlabel'] = {'label':xlabels[1],'fontsize':16}
    params['ylabel'] = {'label':ylabels[1],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});


    ax = axs[(0,0)];
    sindx = 1; sindy = 2; sindz = 0; 
    params1 = {'label':method,'x':('z',sindx),'y':('z',sindy),'z':('z',sindz),'iters':[-1],'legend':lgnd};
    PLTS1.addPlot3D(ax,pen=PENS[method],ins=params1);
    
    params = {};
    params['title'] = {'text':titles[0],'fontsize':20}
    params['xlabel'] = {'label':xlabels[0],'fontsize':16}
    params['ylabel'] = {'label':ylabels[0],'fontsize':16}
    params['zlabel'] = {'label':zlabels[0],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

In [ ]:
print(axs2)

# Fig.14

In [ ]:

# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(1,1)] = [0.55,0.05,0.4,0.35];
grid[(0,1)] = [0.55,0.6,0.4,0.35];
grid[(1,0)] = [0.05,0.05,0.4,0.35];
grid[(0,0)] = [0.05,0.6,0.4,0.35];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [0,3,4,5] # replace with appropriate state indices
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};



state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};


titles = {}; ylabels = {};
titles[2] = 'Flight Path Angle vs Time';
titles[3] = 'Heading vs Time';
titles[0] = 'Attitude vs Time';
titles[1] = 'Velocity vs Time';

ylabels[2] = 'Flight Path Angle [deg]';
ylabels[3] = 'Heading $\psi$ [deg]';
ylabels[0] = 'Attitude [km]';
ylabels[1] = 'Velocity [$m/s$]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig14';
itrs_all = list(range(1000))[2:]; runs = [0];#list(range(1000));
for j,sind in enumerate(state_inds):
    for method in methods:
        aind = state_plot_inds[j];
        ax = axs[aind];
        PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})

        params4 = {'label':method,'x':'t','y':('z',sind),'iters':[-1],'legend':lgnd};
        PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params4); 
        
        params = {};
        params['title'] = {'text':titles[j],'fontsize':20}
        params['xlabel'] = {'label':'Time [s]','fontsize':16}
        params['ylabel'] = {'label':ylabels[j],'fontsize':16}
        params['ticks'] = {'labelsize':20,'width':2};
        PLTS1.setParams(ax,params);
        
        PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});


# Fig.15

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.05,0.05,0.3,0.9];
grid[(0,1)] = [0.4,0.05,0.3,0.9];
grid[(0,2)] = [0.75,0.05,0.3,0.9];

axs = PLTS1.createGrid(fig,grid = grid);

tags = ['max_Q','max_q','max_load']
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};

state_plot_inds = {0:(0,0),1:(0,1),2:(0,2)};
titles = {}; ylabels = {};

titles[0] = 'Heat Rate Constraint';
titles[1] = 'Dynamic Pressure Constraint';
titles[2] = 'Normal Load Constraint';

ylabels[0] = 'Heating Rate [kW/$m^2$]';
ylabels[1] = 'Dynamic Pressure [kPa]';
ylabels[2] = 'Normal Load [$g\'s$]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];
lgnd = 'Fig15';

itrs_all = list(range(1000))[2:]; runs = [0]; #uns = list(range(1000));
for j,tag in enumerate(tags):
    for method in methods:
        ax = axs[state_plot_inds[j]];
        PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})

        # params1 = {'label':'Initial guess','x':'t_nl','y':tag,'iters':[1],'legend':lgnd,};
        # params2 = {'label':'iterations','x':'t_nl','y':tag,'iters':itrs_all,'legend':lgnd};
        params3 = {'label':'Propogated','x':'t_nl','y':tag + '_nl','iters':[-1],'legend':lgnd};
        params4 = {'label':'Optimal Solution','x':'ts','y':tag + '_sub','iters':[-1],'legend':lgnd};
        PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params4); 
    
        params = {};
        params['title'] = {'text':titles[j],'fontsize':20}
        params['xlabel'] = {'label':'Time [s]','fontsize':16}
        params['ylabel'] = {'label':ylabels[j],'fontsize':16}
        params['ticks'] = {'labelsize':20,'width':2};
        PLTS1.setParams(ax,params);
    if j == 2: PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

# Fig.16

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.05,0.05,0.9,0.9];

axs = PLTS1.createGrid(fig,grid = grid);
sinds = [3] ;#tags = ['max_q','max_Q','max_load']
state_names = {0:'Terminal Velocity [m/s]'}
lgnd = 'Fig16';

titles = {}; ylabels = {};
state_plot_inds = {0:(0,0),1:(0,1),2:(0,2)};
titles[0] = 'Penalty weights for constraints';
ylabels[0] = 'Terminal state constraint, quadratic penalty weights';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

itrs_all = list(range(1000))[1:]; runs = list(range(1000));
for j,sind in enumerate(sinds):
    for method in methods:
        ax = axs[state_plot_inds[j]];
        PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})
        params1 = {'label':'Initial guess','tinds':[-1],'y':('z',sind),'iters':itrs_all,'legend':lgnd};
        PLTS1.addPlot2DIter(ax,pen=PENS[method] ,ins=params1); 
    
        params = {};
        params['title'] = {'text':titles[j],'fontsize':20}
        params['xlabel'] = {'label':'Iterations [k]','fontsize':16}
        params['ylabel'] = {'label':ylabels[j],'fontsize':16}
        params['ticks'] = {'labelsize':20,'width':2};
        PLTS1.setParams(ax,params);
    if j == 2: PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});


# Fig.19

In [ ]:
### TO FIX
### ================================================ ###
sind = 0; # STATE INDEX FOR BANK ANGLE 
### ================================================ ###

# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.05,0.05,0.4,0.9];
grid[(0,1)] = [0.50,0.05,0.4,0.9];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [0,1];
state_plot_inds = {0:(0,0),1:(0,1)};
state_names = {sind:'Bank Angle'};

titles = {}; ylabels = {};
titles[0] = 'Bank Angle vs. Time';
titles[1] = 'Angle-of-attack vs. Time';
ylabels[0] = 'Bank Angle, $\sigma$ [deg]';
ylabels[1] = 'Angle-of-attack $\\alpha$ [deg]';

scenarios = ['scenario1'];
methods = ['standard','autotune']; # methods = ['standard','autotune'];

lgnd = 'Fig12';
itrs_all = list(range(1000))[2:]; runs = list(range(1000));
# for j,sind in enumerate(state_inds):
for method in methods:
    PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})
    ax = axs[(0,0)];
    params = {'label':method,'x':'t','y':('u',0),'iters':[-1],'legend':lgnd};
    PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params); 
    # ax = axs[(0,1)];
    # params = {'label':method,'x':'t','y':('u',1),'iters':[-1],'legend':lgnd};
    # PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params); 
    

for j in [0,1]:
    ax = axs[state_plot_inds[j]];
    params = {};
    params['title'] = {'text':titles[j],'fontsize':20}
    params['xlabel'] = {'label':'Time [s]','fontsize':16}
    params['ylabel'] = {'label':ylabels[j],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});



# Fig.20

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid2D = {}; grid3D = {};
grid3D[(0,0)] = [0.05,0.05,0.6,0.9]; 
grid2D[(0,1)] = [0.70,0.05,0.2,0.9];


axs1 = PLTS1.createGrid(fig,grid = grid2D);
# axs2 = PLTS1.createGrid(fig,grid = grid3D,ins={'plt_typ':'3d'});
axs2 = {(0,0): fig.add_axes(grid3D[(0,0)],projection='3d')} # colorbar axis]
axs = {**axs1,**axs2};

state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};

titles = {}; ylabels = {}; xlabels = {}; zlabels = {};
titles[0] = 'Position(3D) vs Time';
titles[1] = 'Position(2D) vs Time';

xlabels[0] = 'Latitude $\phi$ [deg]';
ylabels[0] = 'Longitude $\\theta$ [deg]';
zlabels[0] = 'Altitude $h$ [km]';

ylabels[1] = 'Latitude $\phi$ [deg]';
xlabels[1] = 'Longitude $\\theta$ [deg]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig13'
j = 1;
itrs_all = list(range(1000))[2:]; 
runs = list(range(1000)); #itrs = [-1];
ax = axs[state_plot_inds[j]];
for method in methods:
    ax = axs[(0,1)]; #state_plot_inds[j]];
    sindx = 1; sindy = 2;
    PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})
    params1 = {'label':method,'x':('z',sindx),'y':('z',sindy),'iters':[-1],'legend':lgnd};
    PLTS1.addPlot2D(ax,pen=PENS[method],ins=params1);

    params = {};
    params['title'] = {'text':titles[1],'fontsize':20}
    params['xlabel'] = {'label':xlabels[1],'fontsize':16}
    params['ylabel'] = {'label':ylabels[1],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});


    ax = axs[(0,0)];
    sindx = 1; sindy = 2; sindz = 0; 
    params1 = {'label':method,'x':('z',sindx),'y':('z',sindy),'z':('z',sindz),'iters':[-1],'legend':lgnd};
    PLTS1.addPlot3D(ax,pen=PENS[method],ins=params1);
    
    params = {};
    params['title'] = {'text':titles[0],'fontsize':20}
    params['xlabel'] = {'label':xlabels[0],'fontsize':16}
    params['ylabel'] = {'label':ylabels[0],'fontsize':16}
    params['zlabel'] = {'label':zlabels[0],'fontsize':16}
    params['ticks'] = {'labelsize':20,'width':2};
    PLTS1.setParams(ax,params);
    PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});

# Fig.21

In [ ]:

# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(1,1)] = [0.55,0.05,0.4,0.35];
grid[(0,1)] = [0.55,0.6,0.4,0.35];
grid[(1,0)] = [0.05,0.05,0.4,0.35];
grid[(0,0)] = [0.05,0.6,0.4,0.35];
axs = PLTS1.createGrid(fig,grid = grid);

state_inds = [0,3,4,5] # replace with appropriate state indices
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};



state_plot_inds = {0:(0,0),1:(0,1),2:(1,0),3:(1,1)};


titles = {}; ylabels = {};
titles[2] = 'Flight Path Angle vs Time';
titles[3] = 'Heading vs Time';
titles[0] = 'Attitude vs Time';
titles[1] = 'Velocity vs Time';

ylabels[2] = 'Flight Path Angle [deg]';
ylabels[3] = 'Heading $\psi$ [deg]';
ylabels[0] = 'Attitude [km]';
ylabels[1] = 'Velocity [$m/s$]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];

lgnd = 'Fig14';
itrs_all = list(range(1000))[1:]; runs = list(range(1000));
for j,sind in enumerate(state_inds):
    for method in methods:
        aind = state_plot_inds[j];
        ax = axs[aind];
        PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})

        params4 = {'label':method,'x':'t','y':('z',sind),'iters':[-1],'legend':lgnd};
        PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params4); 
        
        params = {};
        params['title'] = {'text':titles[j],'fontsize':20}
        params['xlabel'] = {'label':'Time [s]','fontsize':16}
        params['ylabel'] = {'label':ylabels[j],'fontsize':16}
        params['ticks'] = {'labelsize':20,'width':2};
        PLTS1.setParams(ax,params);
        
        PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});


# Fig.22

In [ ]:
# grid = PLTS1.specGrid(typ='2x2'); 
fig = plt.figure(figsize=(10,6));
grid = {};
grid[(0,0)] = [0.05,0.05,0.3,0.9];
grid[(0,1)] = [0.4,0.05,0.3,0.9];
grid[(0,2)] = [0.75,0.05,0.3,0.9];

axs = PLTS1.createGrid(fig,grid = grid);

tags = ['max_Q','max_q','max_load']
state_names = {0:'Altitude',1:'Velocity',2:'Flight Path Angle',3:'Heading'};

state_plot_inds = {0:(0,0),1:(0,1),2:(0,2)};
titles = {}; ylabels = {};

titles[0] = 'Heat Rate Constraint';
titles[1] = 'Dynamic Pressure Constraint';
titles[2] = 'Normal Load Constraint';

ylabels[0] = 'Heating Rate [kW/$m^2$]';
ylabels[1] = 'Dynamic Pressure [kPa]';
ylabels[2] = 'Normal Load [$g\'s$]';

scenarios = ['scenario1'];
methods = ['standard','autotune'];
lgnd = 'Fig15';

itrs_all = list(range(1000))[1:]; runs = list(range(1000));
for j,tag in enumerate(tags):
    for method in methods:
        ax = axs[state_plot_inds[j]];
        PLTS1.setCurrent({'scenarios':scenarios,'methods':[method],'runs':runs})

        # params1 = {'label':'Initial guess','x':'t_nl','y':tag,'iters':[1],'legend':lgnd,};
        # params2 = {'label':'iterations','x':'t_nl','y':tag,'iters':itrs_all,'legend':lgnd};
        params3 = {'label':'Propogated','x':'t_nl','y':tag + '_nl','iters':[-1],'legend':lgnd};
        params4 = {'label':'Optimal Solution','x':'ts','y':tag + '_sub','iters':[-1],'legend':lgnd};
        PLTS1.addPlot2D(ax,pen=PENS[method] ,ins=params4); 
    
        params = {};
        params['title'] = {'text':titles[j],'fontsize':20}
        params['xlabel'] = {'label':'Time [s]','fontsize':16}
        params['ylabel'] = {'label':ylabels[j],'fontsize':16}
        params['ticks'] = {'labelsize':20,'width':2};
        PLTS1.setParams(ax,params);
    if j == 2: PLTS1.addLegend(ax,lgnd,ins={'fontsize':14,'loc':'best'});